In [25]:
pip install optuna==4.3.0

,Requirement already satisfied: alembic>=1.5.0 in /usr/local/lib/python3.11/dist-packages (from optuna==4.3.0) (1.15.2)
,Requirement already satisfied: colorlog in /usr/local/lib/python3.11/dist-packages (from optuna==4.3.0) (6.9.0)
,Requirement already satisfied: numpy in /usr/local/lib/python3.11/dist-packages (from optuna==4.3.0) (2.0.2)
,Requirement already satisfied: packaging>=20.0 in /usr/local/lib/python3.11/dist-packages (from optuna==4.3.0) (24.2)
,Requirement already satisfied: sqlalchemy>=1.4.2 in /usr/local/lib/python3.11/dist-packages (from optuna==4.3.0) (2.0.40)
,Requirement already satisfied: tqdm in /usr/local/lib/python3.11/dist-packages (from optuna==4.3.0) (4.67.1)
,Requirement already satisfied: PyYAML in /usr/local/lib/python3.11/dist-packages (from optuna==4.3.0) (6.0.2)
,Requirement already satisfied: Mako in /usr/lib/python3/dist-packages (from alembic>=1.5.0->optuna==4.3.0) (1.1.3)
,Requirement already satisfied: typing-extensions>=4.12 in /usr/local/lib/pyth

In [26]:
import numpy as np
import pandas as pd
import optuna
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score

In [30]:
# 데이터 불러오기
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
submission = pd.read_csv('submission.csv')

In [31]:
# 결측값 처리
categorical_cols = ['분야', '기업가치(백억원)']
for col in categorical_cols:
    train[col] = train[col].fillna(train[col].mode()[0])
    test[col] = test[col].fillna(test[col].mode()[0])

numerical_cols = ['직원 수', '고객수(백만명)']
for col in numerical_cols:
    train[col] = train[col].fillna(train[col].median())
    test[col] = test[col].fillna(test[col].median())

# 범주형 인코딩
categorical_features = ['국가', '분야', '투자단계', '인수여부', '상장여부', '기업가치(백억원)']
for feature in categorical_features:
    le = LabelEncoder()
    train[feature] = le.fit_transform(train[feature].astype(str))
    test[feature] = le.transform(test[feature].astype(str))

In [35]:
# 데이터 분리
X_train = train.drop(columns=['ID', '성공확률'])
y_train = train['성공확률']

# Optuna - XGBoost (n_estimators 고정)
def objective_xgb(trial):
    model = XGBRegressor(
        n_estimators=500,  # 고정
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3),
        max_depth=trial.suggest_int('max_depth', 3, 30),
        min_child_weight=trial.suggest_int('min_child_weight', 1, 10),
        subsample=trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1.0),
        reg_alpha=trial.suggest_float('reg_alpha', 0.0, 1.0),
        reg_lambda=trial.suggest_float('reg_lambda', 0.0, 1.0),
        random_state=42,
        n_jobs=-1
    )
    return cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1).mean()

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=30)

xgb_model = XGBRegressor(**study_xgb.best_params, n_estimators=500, random_state=42, n_jobs=-1)

# Optuna - RandomForest (n_estimators 고정)
def objective_rf(trial):
    model = RandomForestRegressor(
        n_estimators=500,  # 고정
        max_depth=trial.suggest_int('max_depth', 3, 20),
        min_samples_split=trial.suggest_int('min_samples_split', 2, 10),
        min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 4),
        max_features=trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        random_state=42,
        n_jobs=-1
    )
    return cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1).mean()

study_rf = optuna.create_study(direction='maximize')
study_rf.optimize(objective_rf, n_trials=30)

rf_model = RandomForestRegressor(**study_rf.best_params, n_estimators=500, random_state=42, n_jobs=-1)

[I 2025-05-03 09:23:48,915] A new study created in memory with name: no-name-cb716017-632b-4385-9e24-37d1649807d4
,[I 2025-05-03 09:23:52,541] Trial 0 finished with value: -0.2102178828225778 and parameters: {'learning_rate': 0.17633688793761917, 'max_depth': 18, 'min_child_weight': 4, 'subsample': 0.8073445978766802, 'colsample_bytree': 0.906200186998088, 'reg_alpha': 0.001262805170318293, 'reg_lambda': 0.24946247734394655}. Best is trial 0 with value: -0.2102178828225778.
,[I 2025-05-03 09:23:54,769] Trial 1 finished with value: -0.20471899685703693 and parameters: {'learning_rate': 0.02201722716759911, 'max_depth': 7, 'min_child_weight': 6, 'subsample': 0.6183026609260291, 'colsample_bytree': 0.7545418653861415, 'reg_alpha': 0.1905968146832605, 'reg_lambda': 0.44824016723805704}. Best is trial 1 with value: -0.20471899685703693.
,[I 2025-05-03 09:23:56,509] Trial 2 finished with value: -0.20893217004620013 and parameters: {'learning_rate': 0.1669144310841351, 'max_depth': 26, 'min_c

In [37]:
# Soft Voting with RF:XGB = 7:3
voting_model = VotingRegressor(estimators=[
    ('rf', rf_model),
    ('xgb', xgb_model)
], weights=[7, 3])  # RF:XGB = 7:3

# 학습 및 예측
voting_model.fit(X_train, y_train)
test_preds = voting_model.predict(test.drop(columns=['ID']))

# 결과 저장
submission_voting = pd.DataFrame({'ID': test['ID'], '성공확률': test_preds})
submission_voting.to_csv('voting_submission_rf7_xgb3.csv', index=False)

print("✅ RF + XGB Soft Voting (7:3, n_estimators=500 고정) 완료! 'voting_submission_rf7_xgb3.csv' 저장됨.")

✅ RF + XGB Soft Voting (7:3, n_estimators=500 고정) 완료! 'voting_submission_rf7_xgb3.csv' 저장됨.
